# Final comparison (all 4 models)
Add the four `results_*.zip` outputs as inputs (or upload them), plus the RDD2022 dataset for the failure gallery. Owner of integration: whole team.

In [ ]:
import os, subprocess, sys
# Option A (recommended): the final code is pushed to the team repo
if not os.path.exists('repo'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/lakshya1333/Road-damage-detection-deep-learning-project', 'repo'], check=True)
# Option B: if you uploaded the code folder as a Kaggle dataset instead, copy it:  !cp -r /kaggle/input/<your-code-dataset> repo
os.chdir('repo')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics>=8.2', 'segmentation-models-pytorch>=0.3'], check=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Shared manifest: identical split + class mapping for all four models (check the printed class mapping!)
!python -m src.data.build_manifest --out results/data --plot

In [ ]:
import glob, zipfile
for z in glob.glob('/kaggle/input/**/results_*.zip', recursive=True):
    zipfile.ZipFile(z).extractall('results')
# If Kaggle auto-extracted them, copy the folders instead:
!for d in /kaggle/input/*/resnet50 /kaggle/input/*/yolov8 /kaggle/input/*/unet /kaggle/input/*/vit_b16; do [ -d "$d" ] && cp -rn "$d" results/; done
!ls results

In [ ]:
!python -m src.evaluation.compare_models --results results --out results/final --manifest results/data --gallery

In [ ]:
from IPython.display import Image, display
for f in ['per_class_f1.png','confusion_matrices.png','accuracy_by_box_size.png','f1_vs_latency.png']: display(Image('results/final/'+f))

In [ ]:
# Rebuild report + slides with the real numbers
!pip -q install python-docx python-pptx
!python reporting/build_report.py --results results --out ICT4442_Final_Report.docx
!python reporting/build_slides.py --results results --out ICT4442_Final_Presentation.pptx
import shutil; shutil.make_archive('/kaggle/working/final_results','zip','results')